# Spatial Domain Clustering Tutorial

This notebook demonstrates how to perform spatial domain clustering using the imputed gene expression data from SpatialTIP.

## Overview

The workflow includes:
1. Load original and imputed gene expression data
2. Combine imputed values with original non-zero values
3. Apply spatial neighbor smoothing
4. Dimensionality reduction with PCA
5. Clustering with mclust
6. Optional label refinement
7. Visualization and evaluation

## 1. Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
import torch

from utils import Cal_Spatial_Net, mclust_R, refine_label

# Set R environment (required for mclust)
os.environ['R_HOME'] = '/root/miniconda3/envs/spatialtip/lib/R'
os.environ['R_USER'] = '/root/miniconda3/envs/spatialtip/lib/python3.9/site-packages/rpy2'
os.environ['R_LIBS'] = '/root/miniconda3/envs/spatialtip/lib/R/library'

In [ ]:
# Set random seed for reproducibility
seed = 2024
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Load Data

We need two files:
- **Original adata**: Contains spatial coordinates and metadata
- **Imputed adata**: Contains imputed gene expression from SpatialTIP

In [ ]:
# Configure paths
test_sample = 'MISC5'  # Change this to your sample name

# Path to original spatial transcriptomics data
adata_path = f'hest_data/st/{test_sample}.h5ad'

# Path to SpatialTIP imputed data
imputed_path = f'Results/{test_sample}/spatialtip_imputed.h5ad'

# Load data
adata = sc.read_h5ad(adata_path)
adata_SpatialTIP_imputed = sc.read_h5ad(imputed_path)

print(f'Original adata shape: {adata.shape}')
print(f'Imputed adata shape: {adata_SpatialTIP_imputed.shape}')

## 3. Prepare Imputed Expression Matrix

We combine the imputed values with the original non-zero values:
- For genes that were originally expressed (non-zero), we keep the original values
- For genes that were not expressed (zero), we use the imputed values

In [ ]:
# Get raw expression matrix
raw_matrix = adata.X.toarray().copy()
imputed_matrix = adata_SpatialTIP_imputed.X.copy()

# Identify zero positions
zero_mask = raw_matrix == 0

# Keep original non-zero values, set imputed values for zero positions to 0
# (we only trust imputed values for originally expressed genes)
imputed_matrix = np.where(zero_mask, 0, imputed_matrix)

print(f'Number of originally expressed genes per spot: {(~zero_mask).sum(axis=1).mean():.1f} (avg)')

## 4. Spatial Neighbor Smoothing

Spatial smoothing helps denoise the expression by averaging with neighboring spots.

In [ ]:
# Build spatial network
Spatial_Net = Cal_Spatial_Net(adata, rad_cutoff=150, model='Radius')

# Build neighbor dictionary
neighbors = {}
for _, row in Spatial_Net.iterrows():
    spot1, spot2 = row['Spot1'], row['Spot2']
    neighbors.setdefault(spot1, set()).add(spot2)
    neighbors.setdefault(spot2, set()).add(spot1)

print(f'Number of spots with neighbors: {len(neighbors)}')
print(f'Average neighbors per spot: {np.mean([len(v) for v in neighbors.values()]):.1f}')

In [ ]:
# Apply spatial smoothing
smoothed_matrix = imputed_matrix.copy()
for spot, neighbor_spots in neighbors.items():
    neighbor_spots = [int(spot) for spot in neighbor_spots]
    smoothed_matrix[int(spot)] = smoothed_matrix[neighbor_spots].mean(axis=0)

# Create adata with smoothed imputed expression
adata_SpatialTIP = adata.copy()
adata_SpatialTIP.X = smoothed_matrix

## 5. Dimensionality Reduction

We use PCA to reduce the dimensionality of the imputed expression matrix before clustering.

In [ ]:
# Run PCA
n_pcs = 20
adata_SpatialTIP.obsm['X_pca'] = PCA(n_components=n_pcs, random_state=seed).fit_transform(adata_SpatialTIP.X.copy())

print(f'PCA shape: {adata_SpatialTIP.obsm["X_pca"].shape}')

## 6. Clustering with mclust

mclust is a Gaussian mixture model-based clustering method that works well for spatial transcriptomics data.

In [ ]:
# Set number of clusters
# For MISC5 samples: 5 layers (L3-L6 + WM)
# Adjust based on your tissue type
n_clusters = 5

# Run mclust clustering
adata_SpatialTIP = mclust_R(adata_SpatialTIP, num_cluster=n_clusters, used_obsm='X_pca')

print(f'Cluster labels: {adata_SpatialTIP.obs["mclust"].unique()}')

## 7. Label Refinement (Optional)

Label refinement can improve clustering results by considering spatial continuity.

In [ ]:
use_refine = True  # Set to False to skip refinement

if use_refine:
    print('Refining clustering results...')
    refined_labels = refine_label(adata_SpatialTIP, n_neighbors=30, key='mclust')
    adata_SpatialTIP.obs['mclust_refined'] = refined_labels
    cluster_key = 'mclust_refined'
else:
    cluster_key = 'mclust'

## 8. Evaluation (with Ground Truth)

If ground truth labels are available, we can compute the Adjusted Rand Index (ARI).

In [ ]:
# DLPFC sample ID mapping
dataset_id_trans = {
    'MISC1': '151676', 'MISC2': '151675', 'MISC3': '151674', 'MISC4': '151673',
    'MISC5': '151672', 'MISC6': '151671', 'MISC7': '151670', 'MISC8': '151669',
    'MISC9': '151510', 'MISC10': '151509', 'MISC11': '151508', 'MISC12': '151507'
}

# Load ground truth if available
if test_sample in dataset_id_trans:
    ground_truth_path = f'DLPFC/{dataset_id_trans[test_sample]}/{dataset_id_trans[test_sample]}_truth.txt'
    ground_truth_df = pd.read_csv(ground_truth_path, index_col=0, header=None, sep='\t')
    adata_SpatialTIP.obs['ground_truth'] = ground_truth_df.loc[adata_SpatialTIP.obs_names, 1].values
    adata_SpatialTIP.obs['ground_truth'] = adata_SpatialTIP.obs['ground_truth'].astype('category')
    
    # Calculate ARI
    obs_df = adata_SpatialTIP.obs.dropna()
    ari_score = adjusted_rand_score(obs_df['ground_truth'], obs_df[cluster_key])
    print(f'ARI Score: {ari_score:.4f}')
else:
    print('No ground truth available for this sample')

## 9. Visualization

In [ ]:
# Plot clustering results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot clustering result
sc.pl.spatial(adata_SpatialTIP, color=cluster_key, ax=axes[0], show=False, 
              title='SpatialTIP Clustering', size=1.8)

# Plot ground truth if available
if 'ground_truth' in adata_SpatialTIP.obs.columns:
    sc.pl.spatial(adata_SpatialTIP, color='ground_truth', ax=axes[1], show=False,
                  title='Ground Truth', size=1.8)
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Save results
output_dir = f'figures/clustering/{test_sample}'
os.makedirs(output_dir, exist_ok=True)

# Save figure
fig.savefig(f'{output_dir}/clustering_result.png', dpi=300, bbox_inches='tight')

# Save adata with clustering results if needed
adata_SpatialTIP.write(f'Results/{test_sample}/spatialtip_clustered.h5ad')

print(f'Results saved to {output_dir}/')

## Notes

### Set PCA components
- `n_components`: Number of principal components to use (default: 20 components)

### Label Refinement (from GraphST)
- `n_neighbors`: Number of neighbors to consider for refinement (default: 30)
- Can improve results but may over-smooth boundaries